In [0]:
dbutils.widgets.text("file_path", "")
file_path = dbutils.widgets.get("file_path")

# If file_path parameter is not provided, use default test file
if not file_path:
    file_path = "/Volumes/workspace/ds_27/batch_27/IP TEST_Fix DC Greenfield and Supply 1_doublequotes.csv"

# Load the CSV file
employee_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

print(f"Loaded file: {file_path}")
print(f"Row count: {employee_df.count()}")

file_path = "/Volumes/workspace/ds_27/batch_27/IP TEST.csv"


In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/ds_27/batch_27/IP TEST_Fix DC Greenfield and Supply 1_doublequotes.csv")
df.display()

In [0]:
from pyspark.sql.functions import col, lit

# Check if any cell contains double quotes
print("Checking for double quotes in CSV cells...\n")

has_double_quotes = False
columns_with_quotes = []

for column in df.columns:
    # Count rows with double quotes in this column
    count_with_quotes = df.filter(col(column).cast("string").contains('"')).count()
    
    if count_with_quotes > 0:
        has_double_quotes = True
        columns_with_quotes.append(column)
        print(f"Column '{column}': Found {count_with_quotes} rows with double quotes")

if has_double_quotes:
    print(f"\n⚠️ Double quotes detected in {len(columns_with_quotes)} column(s): {columns_with_quotes}")
else:
    print("✓ No double quotes found in any cells")

In [0]:
from pyspark.sql.functions import col

# Find rows with NULL Customer Name (these came from "" in the CSV)
print("Rows where Customer Name is NULL (from \"\" in CSV):\n")
null_customer_rows = df.filter(col("Customer Name").isNull())

if null_customer_rows.count() > 0:
    print(f"Found {null_customer_rows.count()} rows with NULL Customer Name:\n")
    null_customer_rows.select(
        "Customer Acct #", 
        "Customer Name", 
        "Item #", 
        "Warehouse Name"
    ).show(truncate=False)
    
    print("\n⚠️  Note: In the CSV file, these rows have \"\" (empty quoted fields) which Spark interprets as NULL.")
    print("This is standard CSV behavior: \"\" = empty/null value.")
else:
    print("✓ No NULL Customer Names found")

In [0]:
display(df, 1000000)
IP_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/ds_27/batch_27/IP TEST_Fix DC Greenfield and Supply 1_doublequotes.csv")
if employee_df.count() == 0:
    raise Exception("ADF ERROR: IP file is empty. Please upload a valid file with data.")
else:
    print("IP file loaded successfully.")

In [0]:
# to verify if Customer Acct #'  is 100% filled 
# Check total rows
import pyspark.sql.functions as F
from pyspark.sql.functions import col

total_rows = employee_df.count()

# Check rows where Customer Acct # is null or blank
invalid_rows = employee_df.filter(
    col("Customer Acct #").isNull() | (col("Customer Acct #") == "")
).count()

# Validate column completeness
if invalid_rows > 0:
    raise Exception(
        f"ADF ERROR: 'Customer Acct #' column is not 100% filled. "
        f"{invalid_rows} rows have missing or blank values out of {total_rows}."
    )
else:
    print("'Customer Acct #' column is 100% filled.")

In [0]:
from pyspark.sql.functions import col

# Business Rule: At least ONE of (Invoice Units, Invoice Revenue, Invoice Weight (Pounds)) 
# must be present and > 0 in each row

# Find rows where ALL three fields are invalid (null or <= 0)
invalid_rows = employee_df.filter(
    # Invoice Units is invalid: null or <= 0
    (col("Invoice Units").isNull() | (col("Invoice Units") <= 0)) &
    # Invoice Revenue is invalid: null or <= 0
    (col("Invoice Revenue").isNull() | (col("Invoice Revenue") <= 0)) &
    # Invoice Weight (Pounds) is invalid: null or <= 0
    (col("Invoice Weight (Pounds)").isNull() | (col("Invoice Weight (Pounds)") <= 0))
).count()

# Validate that at least one field is present and > 0
if invalid_rows > 0:
    raise Exception(
        f"ADF ERROR: At least one of 'Invoice Units', 'Invoice Revenue', or 'Invoice Weight (Pounds)' "
        f"must be present and greater than 0 in every row. {invalid_rows} invalid rows found."
    )
else:
    print("Validation passed: All rows have at least one valid invoice metric (Units, Revenue, or Weight > 0).")

In [0]:
# TO CHECK if invoice truckload equivalent is presnt if friedt mode is truckload
invalid_rows=employee_df.filter((col("Freight Mode")=="Truckload") & 
                                (col("Invoice Truckload Equivalents").isNull()))

if invalid_rows.count()>0:
    print("invoice truckload equivalent value is not present for truckload freight mode")

else:
    print("invoice truckload equivalent value is  present for truckload freight mode")
    


In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id

# Add row number column (for tracking CSV row positions)
employee_df_with_row = df.withColumn("row_number", monotonically_increasing_id() + 1)

# Find mismatched rows
invalid_rows = employee_df_with_row.filter(
    (col("Freight Mode") == "Truckload") &
    (col("Invoice Truckload Equivalents").isNull())
)

# Check and display row numbers with mismatch
if invalid_rows.count() > 0:
    print("Mismatch found in the following row numbers:")
    invalid_rows.select("row_number", "Freight Mode", "Invoice Truckload Equivalents").show(truncate=False)

    raise Exception("ADF ERROR: Missing 'Invoice Truckload Equivalents' for Truckload freight mode.")
else:
    print("Validation passed: All Truckload rows contain Invoice Truckload Equivalents.")